
`carbamazepine · clobazam · clonazepam · ethosuximide · lamotrigine · levetiracetam · phenobarbital · phenytoin · topiramate · valproate`

### 1. Patient Input (per visit)
Each patient context contains:
- **Demographics**: age, sex
- **Seizure info**: type, frequency, onset age, any status epilepticus
- **Cognitive/developmental status**
- **Risk factors / etiology**
- **Medication history** — what drugs the patient was on before this visit *(format changes per visit — see below)*

### 2. The Reasoning Pipeline
The model is forced to reason **in order** through 7 stages:

| Stage | What it does |
|---|---|
| **0 — Input sanity** | Is the seizure type certain? If not, lean toward broad-spectrum drugs |
| **1 — Seizure compatibility** | Build a candidate shortlist — actively select AND explicitly reject drugs by seizure type |
| **2 — Hard safety gates** | Penalize VPA in pregnancy/liver disease; caution with PHT/VPA in low-albumin states |
| **3 — LMIC anchor** | default to VPA (generalized) or CBZ (focal). Newer drugs (LEV, LTG, TPM) only if older ones are ruled out |
| **4 — Soft tie-breakers** | Side effects, cognitive risk, weight, behavioral concerns |
| **5 — Medication history** | Go through **every prior drug** and explicitly say: continue / stop / no action needed |
| **6 — Mono vs. polytherapy** | Prefer monotherapy; if adding a second drug, justify the choice |

### 3. In-context Drug Knowledge
A reference section covers each drug's niche (focal vs. generalized, adjunct vs. monotherapy) and key contraindications — e.g., CBZ worsens myoclonus, ESM is absence-only, LTG needs slow titration due to rash risk.

### 4. Few-shot Examples (ICL)
Three calibration examples:
- Child with generalized seizures + prior drug history *(Visit 1 style)*
- Adult with focal seizures, treatment-naive
- Child already on VPA with breakthrough seizures *(Visit 2/3 style — polytherapy decision)*

### 5. Output Format (strict, machine-parseable)
```
---SECTION 1: CLINICAL REASONING---
<free-text reasoning following the pipeline>

---SECTION 2: TOP-3 DRUG RECOMMENDATIONS---
Rank 1: <drug> | reason: <1-2 sentences>
Rank 2: <drug> | reason: <1-2 sentences>
Rank 3: <drug> | reason: <1-2 sentences>
```

---

## Visit Structure & Medication History Input

The key difference between visits is **what the medication history section looks like**:

| Visit | Medication history shown to model |
|---|---|
| **Visit 1** | All drugs the patient was ever on, listed as `"previously on"` — no status info. The task is to decide continue / stop / add. |
| **Visit 2** | Full status from Visit 1: `current (ongoing)`, `current (newly started)`, `current (dose changed)`, `previous (stopped)` |
| **Visit 3** | Full status from Visit 2 — same format, one visit further along |

This means the model has **progressively more information** at each visit, which explains the performance trend shown below.

In [4]:
import json, re
import pandas as pd
import numpy as np

# ── Copied verbatim from predict_v2.py ────────────────────────────

DRUG_FEATURE_NAMES = [
    "drug_clobazam", "drug_clonazepam", "drug_valproate", "drug_ethosuximide",
    "drug_levetiracetam", "drug_lamotrigine", "drug_phenobarbital",
    "drug_phenytoin", "drug_topiramate", "drug_carbamazepine",
]

FEATURE_NAMES = [
    "Age_Years", "Sex_Female", "OnsetTimingYears", "SeizureFreq",
    "StatusOrProlonged", "CognitivePriority", "Risk_Factors", "SeizureType",
]

def _extract_status(answer: str) -> str:
    if not answer or answer.strip().lower() in ("not mentioned.", "not mentioned"):
        return None
    parts = answer.split(".")
    last_meaningful = ""
    for part in reversed(parts):
        part = part.strip()
        if part and any(kw in part.lower() for kw in [
            "current", "previous", "unclear", "ongoing",
            "stop", "initiation", "dose_change"
        ]):
            last_meaningful = part
            break
    if not last_meaningful:
        last_meaningful = answer.strip().rstrip(".")
    last_meaningful = re.sub(r'^Visit\s*\d+\s*:\s*', '', last_meaningful).strip()
    return last_meaningful

def _format_status(status: str) -> str:
    s = status.lower()
    if "current" in s:
        use = "current"
    elif "previous" in s:
        use = "previous"
    else:
        return "unclear"
    if "ongoing" in s or "backfilled" in s:
        temp = "ongoing"
    elif "dose_change" in s:
        temp = "dose changed"
    elif "initiation" in s:
        temp = "newly started"
    elif "stop" in s:
        temp = "stopped"
    else:
        temp = None
    if temp:
        return f"{use} ({temp})"
    return use

_MED_LEGEND = (
    "Status key: current (ongoing) = still being taken at prior visit; "
    "current (newly started) = just initiated at prior visit; "
    "current (dose changed) = dose was adjusted at prior visit; "
    "previous (stopped) = discontinued before prior visit."
)

def build_current_medications(visit_name: str, all_visits: dict) -> str:
    visit_num = int(visit_name.split("_")[1])
    found_any = False
    if visit_num == 1:
        lines = ["MEDICATION HISTORY (prior drug exposure — no outcome known):"]
        visit_feats = all_visits.get("Visit_1", {})
        for drug_feat in DRUG_FEATURE_NAMES:
            answer = visit_feats.get(drug_feat, {}).get("Answer", "")
            if not answer or "not mentioned" in answer.lower():
                continue
            status = _extract_status(answer)
            if not status or "initiation" in status.lower():
                continue
            drug_name = drug_feat.replace("drug_", "")
            lines.append(f"- {drug_name}: previously on")
            found_any = True
    else:
        lines = ["MEDICATION HISTORY (status as of prior visit):", _MED_LEGEND]
        prev_visit = f"Visit_{visit_num - 1}"
        prev_feats = all_visits.get(prev_visit, {})
        for drug_feat in DRUG_FEATURE_NAMES:
            answer = prev_feats.get(drug_feat, {}).get("Answer", "")
            if not answer or "not mentioned" in answer.lower():
                continue
            status = _extract_status(answer)
            if status:
                drug_name = drug_feat.replace("drug_", "")
                lines.append(f"- {drug_name}: {_format_status(status)}")
                found_any = True
    if not found_any:
        lines.append("No prior medication information available.")
    return "\n".join(lines)

def build_extracted_content(patient_id: str, visit_name: str, visit_features: dict, all_visits: dict) -> str:
    lines = [f"Patient ID: {patient_id}", f"Visit: {visit_name}", ""]
    visit_1_features = all_visits.get("Visit_1", {})
    lines.append("--- CLINICAL HISTORY ---")
    for feat in FEATURE_NAMES:
        source_features = visit_1_features if feat == "Age_Years" else visit_features
        answer = source_features.get(feat, {}).get('Answer', 'Not available')
        reasoning = source_features.get(feat, {}).get('Reasoning', '')
        lines.append(f"{feat}: {answer} (Additional reasoning: {reasoning})")
    lines.append("")
    lines.append(build_current_medications(visit_name, all_visits))
    return "\n".join(lines)

# ── Load data ──────────────────────────────────────────────────────
with open('csv_reconciled_gpt_oss.json') as f:
    reconciled = json.load(f)
with open('drug_ground_truth.json') as f:
    gold_raw = json.load(f)

VISIT_CSVS = {
    'Visit_1': 'drug/all_drugs/openai_gptoss120b_v1_ext_top3.csv',
    'Visit_2': 'drug/all_drugs/openai_gptoss120b_v2_ext_top3.csv',
    'Visit_3': 'drug/all_drugs/openai_gptoss120b_v3_ext_top3.csv',
}
VISIT_JSONS = {
    'Visit_1': 'drug/all_drugs/openai_gptoss120b_v1_ext_top3.json',
    'Visit_2': 'drug/all_drugs/openai_gptoss120b_v2_ext_top3.json',
    'Visit_3': 'drug/all_drugs/openai_gptoss120b_v3_ext_top3.json',
}

# Load all prediction JSONs (for raw_content / model output)
pred_json = {}
for v, path in VISIT_JSONS.items():
    with open(path) as f:
        pred_json[v] = json.load(f)

# Pick 2 example patients with drug history present in all 3 visits
def pick_examples(n=2):
    results = []
    for pid, visits in reconciled.items():
        has_drugs = False
        for vname in ['Visit_1', 'Visit_2', 'Visit_3']:
            feats = visits.get(vname, {})
            if any(feats.get(dk, {}).get('Answer','') and
                   'not mentioned' not in feats.get(dk,{}).get('Answer','').lower()
                   for dk in DRUG_FEATURE_NAMES):
                has_drugs = True
                break
        if has_drugs and pid in pred_json['Visit_1'] and pid in pred_json['Visit_2'] and pid in pred_json['Visit_3']:
            results.append(pid)
        if len(results) == n:
            break
    return results

EXAMPLE_PIDS = pick_examples()
print("Example patients:", EXAMPLE_PIDS)
print("Setup complete.")

Example patients: ['1_Nanyonga Aisha', '2_Mwania Sheldon']
Setup complete.


In [10]:
def compute_metrics(visit):
    df = pd.read_csv(VISIT_CSVS[visit])
    gold = {}
    for pid, visits in gold_raw.items():
        v = visits.get(visit, {})
        if v:
            gold[pid] = set(d for d, val in v['drug_binary'].items() if val == 1)
    ranked = {str(r['patient_id']): [str(r[f'rank_{k}']).strip().lower() for k in (1,2,3)]
              for _, r in df.iterrows()}
    common = sorted(set(ranked) & set(gold))
    N = len(common)
    t1, t2, t3, jacc = [], [], [], []
    for pid in common:
        g, p = gold[pid], ranked[pid]
        ps = set(p)
        t1.append(int(p[0] in g))
        t2.append(int(bool(set(p[:2]) & g)))
        t3.append(int(bool(ps & g)))
        u = ps | g
        jacc.append(len(ps & g) / len(u) if u else 1.0)
    return dict(N=N, top1=np.mean(t1), top2=np.mean(t2), top3=np.mean(t3),
                jaccard=np.mean(jacc), n1=sum(t1), n2=sum(t2), n3=sum(t3))

print(f"{'═'*52}")
print(f"  {'Visit':<10} {'Top-1':>7} {'Top-2':>7} {'Top-3':>7} {'Jaccard':>9}")
print(f"  {'─'*10} {'─'*7} {'─'*7} {'─'*7} {'─'*9}")
for visit in ['Visit_1', 'Visit_2', 'Visit_3']:
    m = compute_metrics(visit)
    print(f"  {visit:<10} {m['top1']:>7.3f} {m['top2']:>7.3f} {m['top3']:>7.3f} {m['jaccard']:>9.3f}  (n={m['N']})")
print(f"{'═'*52}")

════════════════════════════════════════════════════
  Visit        Top-1   Top-2   Top-3   Jaccard
  ────────── ─────── ─────── ─────── ─────────
  Visit_1      0.611   0.660   0.693     0.239  (n=332)
  Visit_2      0.840   0.870   0.877     0.324  (n=332)
  Visit_3      0.877   0.904   0.925     0.344  (n=332)
════════════════════════════════════════════════════


In [5]:
def show_examples(visit_name):
    sep = "═" * 70
    thin = "─" * 70
    print(f"\n{sep}")
    print(f"  EXAMPLES — {visit_name}")
    print(sep)
    for i, pid in enumerate(EXAMPLE_PIDS, 1):
        all_visits = reconciled[pid]
        visit_features = all_visits.get(visit_name, {})
        model_input = build_extracted_content(pid, visit_name, visit_features, all_visits)
        model_output = pred_json[visit_name].get(pid, {}).get('raw_content', '[not found]')
        print(f"\n{'━'*70}")
        print(f"  Case {i}: {pid}")
        print(f"{'━'*70}")
        print(f"\n[INPUT TO MODEL]\n{thin}")
        print(model_input)
        print(f"\n[MODEL OUTPUT]\n{thin}")
        print(model_output)
        print()

In [6]:
show_examples('Visit_1')


══════════════════════════════════════════════════════════════════════
  EXAMPLES — Visit_1
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Case 1: 1_Nanyonga Aisha
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[INPUT TO MODEL]
──────────────────────────────────────────────────────────────────────
Patient ID: 1_Nanyonga Aisha
Visit: Visit_1

--- CLINICAL HISTORY ---
Age_Years: 5 (Additional reasoning: Age is explicitly stated as 5 years in the visit note.)
Sex_Female: Yes (Additional reasoning: Gender is explicitly indicated as female.)
OnsetTimingYears: Not mentioned. (Additional reasoning: The note explicitly says the age of onset is unknown, providing no duration.)
SeizureFreq: Generalised convulsions 2 days ago (Additional reasoning: The note reports a recent seizure that occurred 2 days prior to the visit.)
StatusOrProlonged: Not mentioned. (Additional reason

In [7]:
show_examples('Visit_2')


══════════════════════════════════════════════════════════════════════
  EXAMPLES — Visit_2
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Case 1: 1_Nanyonga Aisha
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[INPUT TO MODEL]
──────────────────────────────────────────────────────────────────────
Patient ID: 1_Nanyonga Aisha
Visit: Visit_2

--- CLINICAL HISTORY ---
Age_Years: 5 (Additional reasoning: Age is explicitly stated as 5 years in the visit note.)
Sex_Female: Yes (Additional reasoning: Both visits consistently indicate female sex with high confidence.)
OnsetTimingYears: Not mentioned. (Additional reasoning: Both visits did not provide a specific seizure onset age; the only information is that onset is unknown.)
SeizureFreq: Visit 1: Generalised convulsions 2 days before Visit 1. Visit 2: One seizure episode reported. (Additional reasoning: Visit 1 describ

In [8]:
show_examples('Visit_3')


══════════════════════════════════════════════════════════════════════
  EXAMPLES — Visit_3
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Case 1: 1_Nanyonga Aisha
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[INPUT TO MODEL]
──────────────────────────────────────────────────────────────────────
Patient ID: 1_Nanyonga Aisha
Visit: Visit_3

--- CLINICAL HISTORY ---
Age_Years: 5 (Additional reasoning: Age is explicitly stated as 5 years in the visit note.)
Sex_Female: Visit 1: Yes. Visit 2: Yes. Visit 3: Yes. (Additional reasoning: Sex was consistently recorded as female in every visit; the timeline records the repeated confirmation.)
OnsetTimingYears: Not mentioned. (Additional reasoning: None of the three visits supplied information about seizure onset timing; therefore the cumulative answer remains missing.)
SeizureFreq: Visit 1: Generalised convulsions 2 days 

In [9]:
def compute_metrics(visit):
    df = pd.read_csv(VISIT_CSVS[visit])
    gold = {}
    for pid, visits in gold_raw.items():
        v = visits.get(visit, {})
        if v:
            gold[pid] = set(d for d, val in v['drug_binary'].items() if val == 1)
    ranked = {str(r['patient_id']): [str(r[f'rank_{k}']).strip().lower() for k in (1,2,3)]
              for _, r in df.iterrows()}
    common = sorted(set(ranked) & set(gold))
    N = len(common)
    t1, t2, t3, jacc = [], [], [], []
    for pid in common:
        g, p = gold[pid], ranked[pid]
        ps = set(p)
        t1.append(int(p[0] in g))
        t2.append(int(bool(set(p[:2]) & g)))
        t3.append(int(bool(ps & g)))
        u = ps | g
        jacc.append(len(ps & g) / len(u) if u else 1.0)
    return dict(N=N, top1=np.mean(t1), top2=np.mean(t2), top3=np.mean(t3),
                jaccard=np.mean(jacc), n1=sum(t1), n2=sum(t2), n3=sum(t3))

print(f"{'═'*52}")
print(f"  {'Visit':<10} {'Top-1':>7} {'Top-2':>7} {'Top-3':>7} {'Jaccard':>9}")
print(f"  {'─'*10} {'─'*7} {'─'*7} {'─'*7} {'─'*9}")
for visit in ['Visit_1', 'Visit_2', 'Visit_3']:
    m = compute_metrics(visit)
    print(f"  {visit:<10} {m['top1']:>7.3f} {m['top2']:>7.3f} {m['top3']:>7.3f} {m['jaccard']:>9.3f}  (n={m['N']})")
print(f"{'═'*52}")

════════════════════════════════════════════════════
  Visit        Top-1   Top-2   Top-3   Jaccard
  ────────── ─────── ─────── ─────── ─────────
  Visit_1      0.611   0.660   0.693     0.239  (n=332)
  Visit_2      0.840   0.870   0.877     0.324  (n=332)
  Visit_3      0.877   0.904   0.925     0.344  (n=332)
════════════════════════════════════════════════════
